## Conjunto de dados (dataset)

O conjunto de dados base utilizado neste projeto é proveniente do artigo *"Decoding Reddit memes virality"* (Sah & Jordan, 2025).

**Nossa abordagem:** Em vez de utilizarmos a base estruturada (CSV) disponibilizada pelos autores com as *features* já extraídas, optamos pela **ingestão de dados não estruturados**.

* **Volume e estrutura:** Estamos trabalhando com um montante de **~16.000 imagens brutas de memes** (aproximadamente 6GB), divididas em diretórios baseados nas suas comunidades de origem (*subreddits*).
* **Justificativa para a disciplina:** Esses dados exigem o uso de uma arquitetura distribuída (Apache Spark + GPUs via Docker). O objetivo é paralelizar o processamento computacionalmente custoso de aplicar modelos de IA (como OCR) diretamente na fonte para criar a nossa própria base de metadados antes de partirmos para a análise preditiva.

# Validação de pré-processamento (OCR)

Execute este *notebook* **antes** de submeter o *job* completo do Spark. Os objetivos desta etapa inicial são:
1. Confirmar se o modelo (EasyOCR) está operando corretamente e conseguindo acessar a aceleração de GPU do servidor.
2. Realizar uma inspeção qualitativa (*spot-check*) nas imagens e no texto extraído.
3. Validar a expressão regular (Regex) utilizada pelo *pipeline* para estruturar os dados.

## 1 — Validação de disponibilidade das GPUs

Verifica se o ambiente virtual e o PyTorch estão reconhecendo corretamente as placas de vídeo disponíveis para paralelizar e acelerar a inferência do modelo de visão computacional.

In [ ]:
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('Contagem de dispositivos:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

## 2 — Amostragem de caminhos de imagens

Nesta etapa, realizamos uma varredura recursiva nos diretórios para mapear todos os arquivos não estruturados. Em seguida, selecionamos aleatoriamente 5 imagens distribuídas pelos diferentes *subreddits* para comporem a nossa amostra de validação técnica.

In [ ]:
import os
import glob

IMAGES_DIR = '/workspace/data/images'

# Seleciona automaticamente 5 imagens espalhadas pelos subreddits
all_images = glob.glob(f'{IMAGES_DIR}/**/*.jpg', recursive=True) + \
             glob.glob(f'{IMAGES_DIR}/**/*.jpeg', recursive=True) + \
             glob.glob(f'{IMAGES_DIR}/**/*.png', recursive=True)

print(f'Total de imagens encontradas: {len(all_images)}')

# Amostra de até 5 imagens
import random
random.seed(42)
test_paths = random.sample(all_images, min(5, len(all_images)))
print('Caminhos amostrados:')
for p in test_paths:
    print(' ', p)

## 3 — Execução do EasyOCR nas imagens de amostra

Aplicamos o modelo de Reconhecimento Óptico de Caracteres para detectar e extrair os textos dos memes. O bloco abaixo processa a amostra e exibe o grau de confiança (*confidence*) das extrações. Vale ressaltar que a natureza ruidosa dos memes (fontes irregulares, imagens distorcidas) torna essa extração desafiadora.

In [ ]:
import easyocr
import matplotlib.pyplot as plt
from PIL import Image
reader = easyocr.Reader(['en'], gpu=True, download_enabled=False, model_storage_directory='/workspace/data/easyocr_models')
for path in test_paths:
    if not os.path.exists(path):
        print(f'Arquivo não encontrado: {path}')
        continue
    results = reader.readtext(path, detail=1)
    texts = [r[1] for r in results]
    confidences = [round(r[2], 2) for r in results]
    img = Image.open(path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f"{os.path.basename(path)}\n{texts}", fontsize=9)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Extraído : {texts}')
    print(f'Confiança: {confidences}')
    print('---')

## 4 — Validação da expressão regular (Regex)

Garante que o código seja capaz de extrair perfeitamente o nome da comunidade (*subreddit*) e do arquivo original a partir da URI do sistema. Esta etapa é um pré-requisito crítico para que o *pipeline* do Spark consiga realizar o cruzamento (*join*) final com a base de metadados brutos.

In [ ]:
import re

for path in test_paths:
    # O Spark usa URIs file:// internamente
    test_path = path.replace('file://', '')

    subreddit_match = re.search(r'/images/([^/]+)/', test_path)
    filename_match  = re.search(r'/([^/]+\.(?:jpe?g|png))$', test_path)

    subreddit = subreddit_match.group(1) if subreddit_match else 'SEM CORRESPONDÊNCIA'
    filename  = filename_match.group(1)  if filename_match  else 'SEM CORRESPONDÊNCIA'

    print(f'Caminho   : {test_path}')
    print(f'Subreddit : {subreddit}')
    print(f'Arquivo   : {filename}')
    print('---')

## 5 — Inspeção da base de metadados

Por fim, realizamos a leitura inicial das primeiras linhas do arquivo CSV dos metadados (`memes_metadata.csv`) para checar a disponibilidade do arquivo e garantir que as colunas essenciais para o cruzamento dos dados estão estruturadas corretamente.

In [ ]:
import pandas as pd
META_PATH = '/workspace/data/metadata/memes_metadata.csv'
if not os.path.exists(META_PATH):
    print(f'Não encontrado: {META_PATH}')

else:
    df_meta = pd.read_csv(META_PATH, nrows=5)
    print('Colunas:', df_meta.columns.tolist())
    print()
    display(df_meta)